# 개념 코드 실행

각 절의 코드를 셀에서 실행합니다. 첫 환경 셀을 실행한 뒤 필요한 절로 이동합니다. 주 구현 과제는 build-agent.ipynb에서 진행합니다.

In [ ]:
from pathlib import Path
import os, sys, json

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "build_lab" / "materials.py").is_file():
    raise RuntimeError("workshop/notebooks에서 이 노트북을 여십시오.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("실행 위치:", root)

from course.common import get_model

## 도구 입력 조건

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError
from langchain.tools import tool


class TeamInput(BaseModel):
    topic: Literal["정산", "계정"] = Field(description="담당 팀을 찾을 업무명")


@tool(
    "lookup_team",
    description="정산 또는 계정 업무의 담당 팀을 찾습니다. 다른 업무는 지원하지 않습니다.",
    args_schema=TeamInput,
)
def find_team(topic: str) -> str:
    return {"정산": "재무지원팀", "계정": "IT지원팀"}[topic]


print(find_team.name)  # lookup_team
print(find_team.description)
print(find_team.args)
print(find_team.invoke({"topic": "정산"}))  # 재무지원팀

try:
    print(find_team.invoke({"topic": "휴가"}))
except ValidationError:
    print("입력 오류: 정산 또는 계정만 조회할 수 있습니다.")

## 도구의 content와 artifact

In [ ]:
"""모델에 보낼 내용과 프로그램이 사용할 원본을 나눕니다."""
from langchain.tools import tool


@tool(response_format="content_and_artifact")
def lookup_record() -> tuple[str, dict]:
    """정산 담당 팀과 정책 원본을 조회합니다."""
    return "정산 담당은 재무지원팀입니다.", {"id": "P-01", "team": "재무지원팀"}


message = lookup_record.invoke(
    {"name": "lookup_record", "args": {}, "id": "call_1", "type": "tool_call"}
)
print(message.content)
print(message.artifact)

## State의 교체와 누적

In [ ]:
"""State 병합 규칙을 모델 호출 없이 비교합니다."""
from operator import add
from typing import Annotated, TypedDict
from langchain.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    latest: list[str]
    history: Annotated[list[str], add]


def first(state):
    return {"latest": ["조회"], "history": ["조회"]}


def second(state):
    return {"latest": ["초안"], "history": ["초안"]}


graph = StateGraph(State)
graph.add_node("first", first)
graph.add_node("second", second)
graph.add_edge(START, "first")
graph.add_edge("first", "second")
graph.add_edge("second", END)
app = graph.compile()
print(app.invoke({"latest": [], "history": []}))

messages = [HumanMessage(content="계정 담당 팀은?", id="q1")]
messages = add_messages(messages, [AIMessage(content="확인 중입니다.", id="a1")])
messages = add_messages(messages, [AIMessage(content="IT지원팀입니다.", id="a1")])
print([(m.id, m.content) for m in messages])

## 중단과 재개

첫 셀로 그래프를 준비하고 두 번째 셀에서 중단합니다. 승인 요청을 읽은 뒤 세 번째 셀의 decision을 approve 또는 reject로 정해 실행합니다. 다시 비교하려면 준비 셀부터 실행합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from course.graph_lab import InquiryState, review
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

graph = StateGraph(InquiryState)
graph.add_node("review", review)
graph.add_edge(START, "review")
graph.add_edge("review", END)
app = graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "approval-example"}}

In [ ]:
paused = app.invoke({"answer": "재무지원팀에 정산 문의를 전달합니다."}, config)
print(paused["__interrupt__"][0].value)

In [ ]:
decision = "approve"
assert decision in {"approve", "reject"}
resumed = app.invoke(Command(resume=decision), config)
print(resumed["decision"])
print(app.get_state(config).next)

## Skill · 문서 확인과 실제 사용 확인

첫 셀은 모델 없이 Skill 파일을 읽습니다. Jupyter 파일 탐색기에서 skills/policy-answer/SKILL.md를 열어 확인 질문을 추가하고 저장합니다. 다음 모델 셀에서는 정산·계정·없는업무의 답변과 문서 읽기 기록을 비교합니다. API 키가 필요하며 입력마다 호출 비용이 발생합니다. 다른 개념 셀을 전부 실행할 필요는 없습니다.


In [ ]:
skill_path = root / "skills" / "policy-answer" / "SKILL.md"
print(skill_path.read_text(encoding="utf-8"))

**바꿀 것:** 규정이 없을 때 “어떤 업무의 규정을 찾으시나요?”처럼 업무명을 다시 묻도록 절차 한 줄을 구체화합니다. **성공 기준:** 문서가 저장되고, 아래 실행에서 문서 읽기 도구 요청·결과와 답변이 그 절차에 맞는지 확인합니다. 좋은 답변만으로 Skill 사용을 추정하지 않습니다.


In [ ]:
from course.harness_lab import run_deep_agent

topic = "없는업무"  # 정산·계정도 비교합니다.
observed = run_deep_agent(topic, model=get_model())
print(json.dumps(observed, ensure_ascii=False, indent=2))

## 그래프 경로 비교

모델 호출이 없습니다. 두 입력의 visited가 lookup→draft, lookup→ask인지 비교합니다.

In [ ]:
from course.graph_lab import build_graph

app = build_graph()
for contact in ["user@example.test", ""]:
    print(app.invoke({"topic": "정산", "contact": contact}))